In [1]:
import sys
import importlib

sys.path.append('C:/Nacho/Universidad/Prácticas Raman/LIBS_QUANTIFICATION_INTA')
from LIBS_quantification_toolbox import * 

In [6]:
import numpy as np
from sklearn.metrics import r2_score

def replication_r2(spectra):
    """Media y desviación típica de R² entre cada espectro y el promedio."""
    M = np.vstack(spectra)
    mean_spec = M.mean(axis=0)
    r2s = [r2_score(mean_spec, s) for s in M]
    return np.mean(r2s), np.std(r2s)

def coefficient_of_variation(values):
    """Coeficiente de variación: std/mean."""
    v = np.array(values)
    mu = v.mean()
    return np.nan if mu == 0 else v.std() / mu

def compare_preprocessing(carpeta, nombre_base, lb=False, quitar_extremos=True):
    """
    Carga 5 shots de un spot con cargar_espectros_5shots, aplica apply_preprocessing
    con norm_sum=False y True, y muestra en pantalla mean_R2, std_R2 y peak_CV.

    Parámetros
    ----------
    carpeta : str
        Subcarpeta dentro de '../Spectra/'.
    nombre_base : str
        Prefijo común de los archivos (p.ej. 'MX_PY').
    lb : bool
        Pasar a cargar_espectros_5shots.
    quitar_extremos : bool
        Pasar a apply_preprocessing.
    """
    # 1) Cargo los 5 shots + dark + nombres
    ws, n1, n2, n3, n4, n5, band_names = cargar_espectros_5shots(carpeta, nombre_base, lb=lb)

    # empaqueto las 5 réplicas: cada n_i es [UV1,UV2,VIS,NIR]
    raw_reps = [n1, n2, n3, n4, n5]

    # 2) Para cada modo de normalización
    for norm_sum in (False, True):
        processed = []
        for rep in raw_reps:
            wl_proc,spec_proc = apply_preprocessing(rep,ws,
                norm_sum=norm_sum)
            processed.append(spec_proc)

        # 3) Estadísticas R²
        mean_r2, std_r2 = replication_r2(processed)

        # 4) Índice del pico máximo en el primer espectro
        peak_idx = np.argmax(processed[0])
        peak_vals = [s[peak_idx] for s in processed]
        peak_cv = coefficient_of_variation(peak_vals)

        # 5) Mostrar resultados
        method = "sum" if norm_sum else "continuum"
        print(f"\n=== Results for {nombre_base} with norm_sum={norm_sum} ({method}) ===")
        print(f"Mean R² : {mean_r2:.4f}")
        print(f"Std  R² : {std_r2:.4f}")
        print(f"Peak  CV: {peak_cv:.4f}")



In [18]:
compare_preprocessing('20250328_OHO-SN3_LIBS-Quantif_samples/4.Olivino','Olivino_Burst5')


=== Results for Olivino_Burst5 with norm_sum=False (continuum) ===
Mean R² : 0.7741
Std  R² : 0.1559
Peak  CV: 0.7190

=== Results for Olivino_Burst5 with norm_sum=True (sum) ===
Mean R² : 0.7755
Std  R² : 0.1553
Peak  CV: 0.7463
